<a href="https://colab.research.google.com/github/NikoriakViktor/PY-Course-Victor-Nikoriak-22-09-2026/blob/main/module_4/lessons/lesson_32_http_requests/note_lesson_32_http_student.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 🌐 Урок 32 — HTTP: requests, httpx, aiohttp

Диспетчерська «Смачно + Таксі» говорить з рестораном через **API**: питає список ресторанів, перевіряє, чи відкрито, надсилає замовлення. Сьогодні ми — **клієнт**: пишемо код, який робить HTTP-запити, правильно реагує на помилки, не висить на завислому сервері, повторює те, що можна повторити, і робить багато запитів одночасно.

| Розділ | Що всередині | Приклади | Задача |
|---|---|---|---|
| 0 | навчальний API в ноутбуці | 0.1 | |
| 1 | шлях запиту: URL, порт, DNS, TCP, HTTP як текст | 1.1–1.7 | |
| 2 | `requests`: GET, параметри, що насправді пішло на сервер | 2.1–2.5 | |
| 3 | статус-коди і `raise_for_status` | 3.1–3.4 | |
| 4 | POST, JSON, токен, `json=` проти `data=` | 4.1–4.5 | **Задача 1** |
| 5 | тайм-аути, збої мережі, ієрархія винятків | 5.1–5.5 | |
| 6 | повторні спроби з паузою, `Session` | 6.1–6.5 | **Задача 2** |
| 7 | `httpx` і одночасні запити: потоки, `AsyncClient`, `aiohttp`, обмеження | 7.1–7.8 | |
| 8 | свій клієнт до API | 8.1–8.6 | |
| 9 | практика: замовлення з черги | 9.1 | **Задача 3** |
| 10 | справжній API PyPI: офлайн-двійник і живий запит | 10.1–10.3 | |
| 11 | знайди помилку | 11.1–11.2 | |
| 12 | зупиняємо сервер | 12.1 | |

**Як працювати**

- Виконуй клітинки **зверху вниз**. Клітинка з визначенням (`def`, класи, константи) лише створює об'єкт — **запуск завжди в наступній клітинці**.
- Перед **🔮 Прогнозом** спершу відповідай сам, потім розгортай відповідь.
- Інтернет потрібен лише в розділі 10 (і для DNS-запиту в 1.3); без нього ці клітинки друкують пояснення і не падають. Усе інше працює з навчальним сервером прямо в ядрі ноутбука.
- Час у тебе буде трохи іншим, ніж у тексті, — важливе **співвідношення**.
- Теорія й схеми — у книзі: [Урок 32](https://nikoriakviktor.github.io/PY-Course-Victor-Nikoriak-22-09-2026/modules/m4/lesson_32/).

---
## 0. Навчальний API

Сервер диспетчерської — файл `smachno_api.py` поруч із цим ноутбуком. Він написаний на стандартній бібліотеці і працює у фоновому потоці прямо в ядрі ноутбука, без інтернету.

У Colab ноутбук відкривається **без** файлів репозиторію — тоді клітинка нижче завантажить `smachno_api.py` з GitHub. Бібліотеки `requests` і `httpx` у Colab уже є, `aiohttp` клітинка встановить.

| Запит | Що робить |
|---|---|
| `GET /restaurants` | список ресторанів; `?district=Поділ` — фільтр |
| `GET /restaurants/<id>` | один ресторан або 404 |
| `GET /restaurants/<id>/status?delay=0.5` | «чи відкрито?» із затримкою |
| `POST /orders` | створити замовлення; заголовок `Authorization: Bearer smachno-token` |
| `GET /flaky` | перші 2 запити — 503, далі — 200 |
| `GET /slow` | відповідь через 3 секунди |
| `POST /reset` | скинути замовлення й лічильник `/flaky` |

In [ ]:
import importlib.util
import os
import subprocess
import sys
import urllib.request

if not os.path.exists("smachno_api.py"):
    url = ("https://raw.githubusercontent.com/NikoriakViktor/PY-Course-Victor-Nikoriak-22-09-2026/"
           "main/module_4/lessons/lesson_32_http_requests/smachno_api.py")
    urllib.request.urlretrieve(url, "smachno_api.py")

for package in ["requests", "httpx", "aiohttp"]:
    if importlib.util.find_spec(package) is None:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", package], check=True)
print("готово")

In [ ]:
import asyncio
import json
import os
import socket
import time
from urllib.parse import quote, urlsplit

import aiohttp
import httpx
import requests

import smachno_api
from smachno_api import start_server

TOKEN = "smachno-token"          # навчальний; справжній токен — лише зі змінних середовища


def pick_port(preferred):
    """Порт для сервера: той, що в книзі, а якщо він зайнятий — будь-який вільний."""
    with socket.socket() as probe:
        try:
            probe.bind(("127.0.0.1", preferred))
        except OSError:
            probe.bind(("127.0.0.1", 0))      # 0 — «дай будь-який вільний»
        return probe.getsockname()[1]

### Приклад 0.1. Запускаємо сервер

Книга показує адресу `http://127.0.0.1:8031`. Якщо порт 8031 у тебе вже зайнятий іншою програмою, `pick_port` візьме вільний — тому всі приклади нижче беруть адресу зі змінної `BASE`, а не пишуть порт руками. Повторний запуск клітинки не стартує другий сервер.

In [ ]:
if smachno_api._server is None:
    PORT = pick_port(8031)
    BASE = start_server(PORT)
else:                                     # клітинку запустили вдруге — сервер уже працює
    PORT = smachno_api._server.server_address[1]
    BASE = f"http://127.0.0.1:{PORT}"
print(BASE)
print(requests.get(f"{BASE}/restaurants/1", timeout=5).json())

---
## 1. Шлях запиту: URL → порт → DNS → TCP → HTTP

Між `requests.get("https://pypi.org/...")` і словником у Python є кілька кроків. Кожен може зламатися по-своєму — тому розберемо кожен у коді.

```text
URL ──розібрати──► хост + порт + шлях
хост ──DNS──► IP-адреса
IP + порт ──TCP──► з'єднання (для https ще й TLS)
з'єднання ──HTTP──► текст запиту → текст відповіді
текст відповіді ──json()──► dict
```

### Приклад 1.1. URL — з чого складається адреса

In [ ]:
url = urlsplit("https://pypi.org:443/pypi/requests/json?format=short#info")
print("схема:    ", url.scheme)
print("хост:     ", url.hostname)
print("порт:     ", url.port)
print("шлях:     ", url.path)
print("параметри:", url.query)
print("фрагмент: ", url.fragment)

Фрагмент (`#info`) на сервер **не надсилається** — це закладка для браузера. Шлях і параметри потраплять у перший рядок HTTP-запиту.

### Приклад 1.2. Порт за замовчуванням

Якщо порт в адресі не вказано, `url.port` — `None`, а з'єднання все одно йде на конкретний порт: 443 для `https`, 80 для `http`. Напишемо функцію, яка каже, куди насправді піде запит.

In [ ]:
DEFAULT_PORTS = {"http": 80, "https": 443}


def effective_port(address):
    """Порт, на який піде з'єднання: з URL або стандартний для схеми."""
    parts = urlsplit(address)
    return parts.port or DEFAULT_PORTS[parts.scheme]

In [ ]:
for address in ["https://pypi.org/pypi/requests/json", "http://example.com/", f"{BASE}/restaurants"]:
    print(f"{address:40} → порт {effective_port(address)}  (у URL: {urlsplit(address).port})")

**🔮 Прогноз:** Що поверне `effective_port("ftp://files.example.com/")`?

<details>
<summary>Відповідь</summary>

`KeyError: 'ftp'` — у словнику `DEFAULT_PORTS` лише `http` і `https`. HTTP-бібліотеки так само не вміють `ftp://`: `requests` кине `InvalidSchema`.

</details>

### Приклад 1.3. DNS: ім'я → IP-адреси

Мережа доставляє дані за **IP-адресою**. Ім'я в адресу перекладає **DNS** — `socket.getaddrinfo`. Для `pypi.org` потрібен інтернет, тому запит загорнуто в `try`: без мережі клітинка надрукує пояснення.

In [ ]:
def resolve(host, port=443):
    """Усі IP-адреси, які DNS повернув для імені (IPv4 і IPv6)."""
    return sorted({info[4][0] for info in socket.getaddrinfo(host, port, type=socket.SOCK_STREAM)})

In [ ]:
try:
    print("pypi.org →", resolve("pypi.org"))
except socket.gaierror as error:
    print("DNS не відповів (немає інтернету?):", error)
print("localhost →", resolve("localhost", PORT))

`localhost` — це адреса **самого себе** (loopback): `127.0.0.1` (і в IPv6 `::1`). Наш навчальний сервер живе саме там, тому DNS в інтернеті йому не потрібен.

### Приклад 1.4. Порт і TCP: хтось слухає чи ні

IP-адреса доводить дані до комп'ютера, **порт** — до конкретної програми. Спробуємо відкрити TCP-з'єднання з портом нашого сервера і з портом, який ніхто не слухає (`CLOSED_PORT` — щойно перевірений вільний порт).

In [ ]:
CLOSED_PORT = pick_port(8099)      # вільний порт: на ньому гарантовано ніхто не слухає


def can_connect(port, host="127.0.0.1"):
    """Спробувати TCP-з'єднання; повертає 'так' або назву помилки."""
    try:
        with socket.create_connection((host, port), timeout=1):
            return "так"
    except OSError as error:
        return type(error).__name__

In [ ]:
print(f"порт {PORT} (сервер):     ", can_connect(PORT))
print(f"порт {CLOSED_PORT} (нікого немає):", can_connect(CLOSED_PORT))

`ConnectionRefusedError` — комп'ютер відповів «на цьому порту ніхто не слухає». Саме ця помилка ховається всередині `requests.ConnectionError`, коли сервер вимкнений (приклад 5.2).

### Приклад 1.5. HTTP — це текст

Коли TCP-з'єднання є, клієнт надсилає **текст запиту** і читає **текст відповіді**. Надішлемо запит звичайним сокетом, без бібліотек. Функція `send_raw` надсилає готовий текст і читає відповідь до кінця; `raw_get` складає текст GET-запиту.

In [ ]:
def send_raw(request_text):
    """Надіслати текст HTTP-запиту через сокет і повернути сирі байти відповіді."""
    with socket.create_connection(("127.0.0.1", PORT), timeout=5) as sock:
        sock.sendall(request_text.encode("utf-8"))
        response = b""
        while chunk := sock.recv(1024):
            response += chunk
    return response


def raw_get(path):
    request_text = (
        f"GET {path} HTTP/1.1\r\n"
        f"Host: 127.0.0.1:{PORT}\r\n"
        "Connection: close\r\n"
        "\r\n"
    )
    return send_raw(request_text)

In [ ]:
print(raw_get("/restaurants/3").decode("utf-8"))

Перший рядок — версія і **статус-код** (`200 OK`), далі заголовки по одному в рядку, порожній рядок і **тіло** — JSON. `Connection: close` просить сервер закрити з'єднання після відповіді, тому цикл `recv` закінчується.

**🔮 Прогноз:** Що зміниться у першому рядку відповіді, якщо попросити `/restaurants/42`?

<details>
<summary>Відповідь</summary>

`HTTP/1.1 404 Not Found` замість `HTTP/1.1 200 OK`, а в тілі — `{"error": "ресторан 42 не знайдено"}`. Перевір: `print(raw_get("/restaurants/42").decode())`.

</details>

### Приклад 1.6. Розбираємо відповідь: статус, заголовки, тіло

Бібліотеки роблять саме це — ділять текст відповіді на частини. Зробимо руками і перевіримо `Content-Length`: це довжина тіла в **байтах**, а не в символах.

In [ ]:
def split_response(raw):
    """Сирі байти відповіді → (рядок статусу, словник заголовків, тіло-байти)."""
    head, body = raw.split(b"\r\n\r\n", 1)
    status_line, *header_lines = head.decode("latin-1").split("\r\n")
    headers = dict(line.split(": ", 1) for line in header_lines)
    return status_line, headers, body

In [ ]:
status_line, headers, body = split_response(raw_get("/restaurants/1"))
text = body.decode("utf-8")
print(status_line)
print(headers["Content-Type"])
print("Content-Length:", headers["Content-Length"], "| байтів у тілі:", len(body), "| символів:", len(text))
print(json.loads(text))

Байтів більше, ніж символів: кожна українська літера в UTF-8 займає 2 байти (урок 15). Тому `Content-Length` завжди рахують від **закодованого** тіла.

### Приклад 1.7. POST руками: тіло, `Content-Type`, токен

POST відрізняється тим, що в запиті теж є тіло. Сервер має знати, де воно закінчується, — тому клієнт пише `Content-Length`. Токен іде заголовком `Authorization`.

In [ ]:
order_json = json.dumps({"restaurant_id": 2, "customer": "Олег", "total": 315}, ensure_ascii=False)
order_bytes = order_json.encode("utf-8")
post_text = (
    "POST /orders HTTP/1.1\r\n"
    f"Host: 127.0.0.1:{PORT}\r\n"
    f"Authorization: Bearer {TOKEN}\r\n"
    "Content-Type: application/json\r\n"
    f"Content-Length: {len(order_bytes)}\r\n"
    "Connection: close\r\n"
    "\r\n"
    + order_json
)
status_line, headers, body = split_response(send_raw(post_text))
print(status_line)
print(body.decode("utf-8"))

`201 Created` — замовлення створено. Усе, що далі роблять `requests`, `httpx` і `aiohttp`, — це складання такого тексту, кодування параметрів і тіла, розбір відповіді, перевикористання з'єднань і HTTPS-шифрування.

---
## 2. requests: перший запит

[`requests`](https://requests.readthedocs.io/) — найпопулярніша HTTP-бібліотека Python: простий, синхронний клієнт (програма чекає на відповідь).

### Приклад 2.1. GET і JSON-відповідь

In [ ]:
response = requests.get(f"{BASE}/restaurants", timeout=5)
print(response.status_code)
print(response.headers["Content-Type"])
for restaurant in response.json():
    print(restaurant["id"], restaurant["name"], restaurant["district"])

- `timeout=5` — чекати відповідь не довше 5 секунд. Чому це обов'язково — у розділі 5.
- `response.json()` — розібрати тіло як JSON: тут список словників.

### Приклад 2.2. Що ще є у відповіді

`response.headers` — словник, **нечутливий до регістру**; `response.text` — тіло рядком; `response.elapsed` — скільки йшла відповідь.

In [ ]:
response = requests.get(f"{BASE}/restaurants/4", timeout=5)
print(response.headers["content-type"] == response.headers["Content-Type"])
print(repr(response.text))
print(type(response.json()), response.json()["name"])
print(f"відповідь за {response.elapsed.total_seconds() * 1000:.1f} мс")

### Приклад 2.3. Параметри запиту

Фільтр передаємо **параметрами** — частиною URL після `?`. Склеювати їх руками не треба: `requests` сам закодує кирилицю.

In [ ]:
response = requests.get(f"{BASE}/restaurants", params={"district": "Поділ"}, timeout=5)
print(response.url)
print([restaurant["name"] for restaurant in response.json()])
print("те саме руками:", quote("Поділ"))

`%D0%9F%D0%BE…` — байти UTF-8 слова «Поділ», записані через `%`: у URL дозволені лише латинські літери, цифри й кілька знаків.

### Приклад 2.4. Функція «назви ресторанів району»

Один запит, фільтр — параметром, обов'язково `timeout`, перевірка статусу — і список назв у тому порядку, як повертає API.

In [ ]:
def names_in(district):
    response = requests.get(f"{BASE}/restaurants", params={"district": district}, timeout=5)
    response.raise_for_status()
    return [restaurant["name"] for restaurant in response.json()]

In [ ]:
for district in ["Оболонь", "Центр", "Троєщина"]:
    print(f"{district:9} →", names_in(district))

assert names_in("Оболонь") == ["Суші Оболонь"]
assert names_in("Центр") == ["Вареники 24/7", "Шаурма Центр"]
assert names_in("Троєщина") == []

Для району без ресторанів сервер повертає `200` і порожній список — це не помилка: запит правильний, просто даних немає.

### Приклад 2.5. Що насправді пішло на сервер

`response.request` — запит, який бібліотека відправила. Корисно подивитись, що вона додала сама.

In [ ]:
sent = requests.get(f"{BASE}/restaurants", params={"district": "Центр"}, timeout=5).request
print(sent.method, sent.url)
for name, value in sent.headers.items():
    print(f"  {name}: {value}")

`User-Agent: python-requests/…` — так сервер бачить, хто до нього прийшов. `Accept-Encoding: gzip, deflate` — «можу прийняти стиснуту відповідь», `Connection: keep-alive` — «не закривай з'єднання, я ще прийду».

---
## 3. Статус-коди

| Клас | Значення | Приклади |
|---|---|---|
| `2xx` | успіх | `200 OK`, `201 Created` |
| `3xx` | перенаправлення | бібліотеки йдуть за ним самі |
| `4xx` | помилка **клієнта**: виправ запит | `400`, `401`, `404`, `422` |
| `5xx` | помилка **сервера**: запит міг бути правильним | `500`, `503` |

### Приклад 3.1. 404 — це не виняток

`requests` **не** викидає виняток на 4xx/5xx: запит відбувся, відповідь прийшла — просто з іншим `status_code`.

In [ ]:
response = requests.get(f"{BASE}/restaurants/42", timeout=5)
print(response.status_code, response.ok, response.json())

### Приклад 3.2. `raise_for_status()` перетворює 4xx/5xx на виняток

In [ ]:
try:
    response.raise_for_status()
except requests.HTTPError as error:
    print("HTTPError:", error)
    print("код у винятку:", error.response.status_code)

print(requests.get(f"{BASE}/restaurants/1", timeout=5).raise_for_status())   # для 2xx — нічого

Для `2xx` метод нічого не робить і повертає `None`. Через виняток відповідь доступна як `error.response` — з неї можна дістати код і тіло.

### Приклад 3.3. «Ресторан або `None`»

Часто 404 — очікувана ситуація («такого немає»), а 5xx — справжня аварія. Тоді 404 обробляємо окремо **до** `raise_for_status()`, а решту помилок пускаємо далі винятком.

In [ ]:
def restaurant_or_none(restaurant_id):
    response = requests.get(f"{BASE}/restaurants/{restaurant_id}", timeout=5)
    if response.status_code == 404:
        return None
    response.raise_for_status()
    return response.json()

In [ ]:
print(restaurant_or_none(4))
print(restaurant_or_none(42))
assert restaurant_or_none(4) == {"id": 4, "name": "Вареники 24/7", "district": "Центр"}
assert restaurant_or_none(42) is None

### Приклад 3.4. Клас статус-коду — за першою цифрою

In [ ]:
STATUS_CLASSES = {2: "успіх", 3: "перенаправлення", 4: "помилка клієнта — виправ запит",
                  5: "помилка сервера — можна спробувати пізніше"}


def status_class(code):
    return STATUS_CLASSES.get(code // 100, "невідомо")

In [ ]:
for code in [200, 201, 302, 401, 404, 422, 503]:
    print(code, "—", status_class(code))
print(requests.get(f"{BASE}/menu", timeout=5).status_code, "—", status_class(404))

Правило, яке знадобиться в розділі 6: **4xx повторювати марно** — сервер знову скаже те саме. **5xx інколи минає**, якщо трохи почекати.

---
## 4. POST, JSON і токен

`requests.post(url, json=дані, headers={...}, timeout=5)`: `json=` сам серіалізує словник і додає `Content-Type: application/json`.

| Код | Що означає тут |
|---|---|
| `201` | замовлення створено, у тілі — `id` |
| `401` | немає або неправильний токен |
| `422` | дані не пройшли перевірку: `{"errors": [...]}` |

### Приклад 4.1. Без токена, з поганими даними, правильно

In [ ]:
requests.post(f"{BASE}/reset", timeout=5)          # номери замовлень знову з 101
order = {"restaurant_id": 1, "customer": "Олена", "total": 420}
headers = {"Authorization": f"Bearer {TOKEN}"}

no_token = requests.post(f"{BASE}/orders", json=order, timeout=5)
print(no_token.status_code, no_token.json())

bad_data = requests.post(f"{BASE}/orders", json={"restaurant_id": 9, "total": -5}, headers=headers, timeout=5)
print(bad_data.status_code, bad_data.json())

created = requests.post(f"{BASE}/orders", json=order, headers=headers, timeout=5)
print(created.status_code, created.json())

`401` — сервер не знає, хто ти. `422` — знає, але дані не пройшли перевірку, і сервер пояснює **всі** проблеми одразу. `201 Created` — створено, у відповіді — `id`.

### Приклад 4.2. Що пішло в тілі запиту

In [ ]:
print(created.request.headers["Content-Type"])
print(created.request.headers["Content-Length"])
print(created.request.body)

`json=` перетворив словник на JSON-рядок, закодував у байти й сам порахував `Content-Length` — те, що ми робили руками в прикладі 1.7. Кирилиця в тілі записана як `О…`: так `json.dumps` робить за замовчуванням, і це правильний JSON.

### Приклад 4.3. `json=` проти `data=`

`data=словник` — це **форма** (`application/x-www-form-urlencoded`), як у HTML-формі, а не JSON. Підготуємо обидва запити без відправлення (`Request(...).prepare()`) і подивимось різницю, а потім відправимо форму нашому JSON-серверу.

In [ ]:
as_json = requests.Request("POST", f"{BASE}/orders", json=order).prepare()
as_form = requests.Request("POST", f"{BASE}/orders", data=order).prepare()
print("json=:", as_json.headers["Content-Type"], "|", as_json.body)
print("data=:", as_form.headers["Content-Type"], "|", as_form.body)

form_response = requests.post(f"{BASE}/orders", data=order, headers=headers, timeout=5)
print(form_response.status_code, form_response.json())

Сервер чекає JSON, а отримав `restaurant_id=1&customer=...` — і чесно відповів `400`. Якщо API в документації пише «тіло — JSON», передаємо `json=`.

### Приклад 4.4. Токен — зі змінної середовища

`smachno-token` — навчальний. Справжній токен — як пароль: **не** пишемо в код і не комітимо в git. Беремо зі змінної середовища (урок 13), а для навчання — запасне значення.

In [ ]:
token = os.environ.get("SMACHNO_TOKEN", "smachno-token")
print("токен зі змінної середовища" if "SMACHNO_TOKEN" in os.environ else "змінної немає — навчальний токен")
print(requests.post(f"{BASE}/orders", json=order, headers={"Authorization": f"Bearer {token}"}, timeout=5).status_code)

### Задача 1. `create_order(customer, total, restaurant_id=1, token=TOKEN)`

Напиши функцію, яка надсилає замовлення на `POST {BASE}/orders` і повертає пару `(status_code, тіло як dict)`.

- тіло — JSON `{"restaurant_id": …, "customer": …, "total": …}` (параметр `json=`);
- заголовок `Authorization: Bearer <token>`; якщо `token` — `None`, заголовок **не** надсилається;
- обов'язково `timeout`.

Перевірка — у наступній клітинці.

In [ ]:
def create_order(customer, total, restaurant_id=1, token=TOKEN):
    # YOUR CODE HERE
    pass


In [ ]:
requests.post(f"{BASE}/reset", timeout=5)
result = create_order("Олена", 420, token=None)
assert result is not None, "функція має повертати пару (status_code, тіло)"
status, body = result
assert status == 401, f"без токена очікували 401, отримали {status}"
status, body = create_order("Олена", 420, token="wrong")
assert status == 401, "неправильний токен — теж 401"
status, body = create_order("", -1, restaurant_id=99)
assert status == 422 and len(body["errors"]) == 3, body
status, body = create_order("Олена", 420)
assert status == 201 and body["id"] == 101 and body["status"] == "new", body
print(status, body)
print("✅ Задача 1 розв'язана")

### Приклад 4.5. POST не ідемпотентний

Надішлемо **те саме** замовлення двічі поспіль.

In [ ]:
requests.post(f"{BASE}/reset", timeout=5)
same = {"restaurant_id": 3, "customer": "Марко", "total": 300}
first = requests.post(f"{BASE}/orders", json=same, headers=headers, timeout=5).json()
second = requests.post(f"{BASE}/orders", json=same, headers=headers, timeout=5).json()
print(first["id"], second["id"])

**🔮 Прогноз:** Що станеться, якщо програма автоматично повторить `POST /orders`, коли перший запит «не дочекався відповіді»?

<details>
<summary>Відповідь</summary>

Перший запит міг уже створити замовлення — сервер просто не встиг відповісти. Повтор створить **дубль**, як `101` і `102` вище: клієнт отримає два обіди. Тому `POST` автоматично не повторюють (розділ 6).

</details>

---
## 5. Тайм-аути і збої мережі

Без `timeout` `requests` чекає **вічно**. Endpoint `/slow` відповідає через 3 секунди.

### Приклад 5.1. Сервер повільний — `ReadTimeout`

In [ ]:
start = time.perf_counter()
try:
    requests.get(f"{BASE}/slow", timeout=1)
except requests.Timeout as error:
    print(type(error).__name__, "—", error)
print(f"чекали {time.perf_counter() - start:.1f} с, а не 3")

### Приклад 5.2. Сервера немає — `ConnectionError`

На `CLOSED_PORT` ніхто не слухає (приклад 1.4).

In [ ]:
try:
    requests.get(f"http://127.0.0.1:{CLOSED_PORT}/restaurants", timeout=1)
except requests.ConnectionError as error:
    print(type(error).__name__)
    print("причина всередині:", type(error.args[0].reason).__name__)

Всередині `requests.ConnectionError` — та сама відмова TCP з прикладу 1.4 (`NewConnectionError` з `ConnectionRefusedError`): бібліотека загортає низькорівневі помилки у свої.

### Приклад 5.3. Ієрархія винятків `requests`

Усі винятки `requests` — нащадки `requests.RequestException`. Тому `except requests.RequestException` ловить будь-який збій запиту, а вужчі класи — конкретні ситуації.

In [ ]:
for cls in [requests.ConnectionError, requests.Timeout, requests.ReadTimeout, requests.ConnectTimeout,
            requests.HTTPError, requests.JSONDecodeError]:
    parents = [parent.__name__ for parent in cls.__mro__[1:] if parent.__module__.startswith("requests")]
    print(f"{cls.__name__:16} ← {', '.join(parents)}")

`ConnectTimeout` — нащадок і `ConnectionError`, і `Timeout`: «не встигли з'єднатися» — це водночас і проблема з'єднання, і тайм-аут.

| Виняток | Коли | Що робити |
|---|---|---|
| `ConnectionError` | сервер вимкнений, немає мережі | повторити пізніше |
| `Timeout` | не встигли з'єднатися або дочекатися | повторити, якщо запит ідемпотентний |
| `HTTPError` | 4xx/5xx — лише після `raise_for_status()` | 4xx — виправити, 5xx — можна повторити |
| `JSONDecodeError` | `.json()` для тіла, яке не JSON | перевірити статус і `Content-Type` |

### Приклад 5.4. Тайм-аут парою: з'єднання і читання

`timeout=(3, 0.5)` — 3 секунди на з'єднання і 0,5 на очікування відповіді.

In [ ]:
start = time.perf_counter()
try:
    requests.get(f"{BASE}/slow", timeout=(3, 0.5))
except requests.ReadTimeout:
    print(f"ReadTimeout через {time.perf_counter() - start:.1f} с: з'єднались швидко, відповіді не дочекались")

### Приклад 5.5. «JSON або `None`, якщо сервер не встиг»

In [ ]:
def get_json_or_none(path, timeout):
    try:
        response = requests.get(f"{BASE}{path}", timeout=timeout)
    except requests.Timeout:
        return None
    response.raise_for_status()
    return response.json()

In [ ]:
start = time.perf_counter()
print(get_json_or_none("/slow", timeout=0.5), f"за {time.perf_counter() - start:.1f} с")
print(get_json_or_none("/restaurants/2", timeout=5))
assert get_json_or_none("/restaurants/2", timeout=5)["name"] == "Піца Поділ"

Тайм-аут ми перетворили на `None`, а 404 чи 500 — ні: вони вилетять `HTTPError`. Ловимо лише те, що вміємо обробити.

---
## 6. Повторні спроби з паузою

Сервіс інколи коротко «лежить». Endpoint `/flaky` імітує це: два перші запити отримують `503`. Повторюємо, **подвоюючи паузу** (exponential backoff), щоб не добивати перевантажений сервер.

Три правила:

1. повторюй лише те, що може минути: 5xx, тайм-аут, немає з'єднання; **4xx — ні**;
2. обмежуй кількість спроб і збільшуй паузу;
3. не повторюй автоматично неідемпотентні запити (`POST`, приклад 4.5).

### Приклад 6.1. Повтор із книги: друкуємо кожну спробу

In [ ]:
def get_with_retry_verbose(url, attempts=4, pause=0.2):
    for attempt in range(1, attempts + 1):
        try:
            response = requests.get(url, timeout=2)
            if response.status_code < 500:
                return response
            print(f"спроба {attempt}: {response.status_code}, чекаю {pause} с")
        except (requests.ConnectionError, requests.Timeout) as error:
            print(f"спроба {attempt}: {type(error).__name__}, чекаю {pause} с")
        time.sleep(pause)
        pause *= 2
    raise RuntimeError(f"{url}: немає відповіді після {attempts} спроб")

In [ ]:
requests.post(f"{BASE}/reset", timeout=5)
response = get_with_retry_verbose(f"{BASE}/flaky")
print(response.status_code, response.json())

### Приклад 6.2. Спроби скінчились

Ті самі 503, але лише дві спроби — функція здається і кидає `RuntimeError`. А на вимкнений сервер вона повторює `ConnectionError`.

In [ ]:
requests.post(f"{BASE}/reset", timeout=5)
for url in [f"{BASE}/flaky", f"http://127.0.0.1:{CLOSED_PORT}/"]:
    try:
        get_with_retry_verbose(url, attempts=2, pause=0.05)
    except RuntimeError as error:
        print("RuntimeError:", error)

### Приклад 6.3. Як росте пауза

In [ ]:
def backoff_delays(attempts, pause):
    """Паузи між спробами: після кожної невдалої, крім останньої."""
    return [pause * 2 ** n for n in range(attempts - 1)]

In [ ]:
print(backoff_delays(4, 0.2))
print(backoff_delays(8, 0.5), "→ разом", sum(backoff_delays(8, 0.5)), "с")

Подвоєння швидко стає довгим: 8 спроб від 0,5 с — понад хвилину очікування. Тому спроб мало (3–5), а для довшої недоступності — черга завдань (уроки 49–50).

### Задача 2. `get_with_retry(path, attempts=4, pause=0.05)`

Функція робить `GET {BASE}{path}` (з `timeout`) і повертає пару `(response, скільки спроб зроблено)`.

- відповідь з кодом `< 500` — повертай **одразу** (4xx не повторюємо);
- 5xx — чекай `pause`, подвоюй паузу і пробуй знову;
- якщо всі спроби дали 5xx — поверни **останню** відповідь (а не виняток).

Перевірка — у наступній клітинці.

In [ ]:
def get_with_retry(path, attempts=4, pause=0.05):
    # YOUR CODE HERE
    pass


In [ ]:
requests.post(f"{BASE}/reset", timeout=5)
result = get_with_retry("/flaky")
assert result is not None, "функція має повертати пару (response, кількість спроб)"
response, tries = result
assert response.status_code == 200 and tries == 3, (response.status_code, tries)

response, tries = get_with_retry("/restaurants/42")
assert response.status_code == 404 and tries == 1, "4xx не повторюємо"

requests.post(f"{BASE}/reset", timeout=5)
response, tries = get_with_retry("/flaky", attempts=2)
assert response.status_code == 503 and tries == 2, "після останньої спроби повертаємо останню відповідь"
print("✅ Задача 2 розв'язана")

### Приклад 6.4. `Session`: спільні заголовки і одне з'єднання

Кожен `requests.get(...)` відкриває нове TCP-з'єднання. `requests.Session` **тримає з'єднання відкритим** і зберігає спільні заголовки — токен достатньо вказати один раз.

In [ ]:
requests.post(f"{BASE}/reset", timeout=5)
with requests.Session() as session:
    session.headers["Authorization"] = f"Bearer {TOKEN}"
    for customer, total in [("Андрій", 250), ("Марта", 610)]:
        response = session.post(f"{BASE}/orders", json={"restaurant_id": 4, "customer": customer, "total": total},
                                timeout=5)
        print(response.status_code, response.json()["id"], response.json()["customer"])

### Приклад 6.5. Наскільки `Session` швидша

100 запитів без сесії і з сесією.

In [ ]:
def timed(label, fn, repeat=100):
    start = time.perf_counter()
    for _ in range(repeat):
        fn()
    print(f"{label}: {time.perf_counter() - start:.2f} с")

In [ ]:
timed("без Session", lambda: requests.get(f"{BASE}/restaurants/1", timeout=5))
with requests.Session() as session:
    timed("з Session  ", lambda: session.get(f"{BASE}/restaurants/1", timeout=5))

Сервер тут на тому самому комп'ютері, тож різниця невелика. Зі справжнім сервером через інтернет вона значно більша: нове HTTPS-з'єднання — це кілька обмінів повідомленнями (TCP + TLS) ще до першого байта запиту.

---
## 7. httpx і одночасні запити

[`httpx`](https://www.python-httpx.org/) майже повторює інтерфейс `requests`, але має **тайм-аут за замовчуванням** (5 с), `base_url` і **асинхронний** клієнт.

### Приклад 7.1. `httpx.Client` — аналог `Session`

In [ ]:
with httpx.Client(base_url=BASE, timeout=5) as client:
    response = client.get("/restaurants", params={"district": "Центр"})
    print(response.status_code, response.http_version)
    print([restaurant["name"] for restaurant in response.json()])
    try:
        client.get("/restaurants/42").raise_for_status()
    except httpx.HTTPStatusError as error:
        print(type(error).__name__, error.response.status_code, error.response.json())

Назви винятків інші: `httpx.HTTPStatusError`, `httpx.TimeoutException`, `httpx.ConnectError`; спільний предок мережевих помилок — `httpx.TransportError`.

### Приклад 7.2. Тайм-аут за замовчуванням

In [ ]:
with httpx.Client() as client:
    print("httpx:", client.timeout)
start = time.perf_counter()
try:
    httpx.get(f"{BASE}/slow", timeout=0.5)
except httpx.TimeoutException as error:
    print(type(error).__name__, f"через {time.perf_counter() - start:.1f} с")

У `httpx` забутий `timeout` — це 5 секунд, а не «вічно», як у `requests`. Але явний `timeout` все одно краще: одразу видно, скільки ми готові чекати.

### Приклад 7.3. По черзі: 5 статусів по 0,5 секунди

In [ ]:
start = time.perf_counter()
statuses = []
for restaurant_id in range(1, 6):
    response = requests.get(f"{BASE}/restaurants/{restaurant_id}/status", params={"delay": 0.5}, timeout=5)
    statuses.append(response.json())
print(f"по черзі: {time.perf_counter() - start:.1f} с", [status["open"] for status in statuses])

2,5 секунди програма просто **чекала**. Це I/O-bound задача (урок 28): поки чекаємо одну відповідь, можна відправити інші.

### Приклад 7.4. Потоки: синхронний `requests` у пулі

In [ ]:
from concurrent.futures import ThreadPoolExecutor


def fetch_status_sync(restaurant_id):
    response = requests.get(f"{BASE}/restaurants/{restaurant_id}/status", params={"delay": 0.5}, timeout=5)
    response.raise_for_status()
    return response.json()

In [ ]:
start = time.perf_counter()
with ThreadPoolExecutor(max_workers=5) as pool:
    statuses = list(pool.map(fetch_status_sync, range(1, 6)))
print(f"потоки: {time.perf_counter() - start:.1f} с", [status["id"] for status in statuses])

### Приклад 7.5. `httpx.AsyncClient` + `asyncio.gather`

У Jupyter цикл подій уже працює, тому корутину запускаємо просто `await корутина()` — без `asyncio.run` (урок 28). Передамо ідентифікатори у зворотному порядку.

In [ ]:
async def check_all(ids, delay=0.5):
    async with httpx.AsyncClient(base_url=BASE, timeout=5) as client:
        async def fetch(restaurant_id):
            response = await client.get(f"/restaurants/{restaurant_id}/status", params={"delay": delay})
            response.raise_for_status()
            return response.json()

        return await asyncio.gather(*(fetch(restaurant_id) for restaurant_id in ids))

In [ ]:
start = time.perf_counter()
statuses = await check_all([5, 4, 3, 2, 1])
elapsed = time.perf_counter() - start
print(f"одночасно: {elapsed:.1f} с", [status["id"] for status in statuses])
assert [status["id"] for status in statuses] == [5, 4, 3, 2, 1]
assert elapsed < 1.5, "запити йшли по черзі?"

- `await client.get(...)` — «відправ запит і, поки чекаєш, віддай керування іншим корутинам».
- `gather` повертає результати **в порядку переданих задач** (5, 4, 3, 2, 1), а не в порядку, в якому прийшли відповіді.

**🔮 Прогноз:** Скільки часу займуть 20 запитів з `delay=0.5` через `check_all`? А 20 по черзі?

<details>
<summary>Відповідь</summary>

Приблизно 0,5 секунди одночасно (усі чекають разом) проти 10 секунд по черзі. На чужому сервері стільки одночасних запитів треба обмежувати — приклад 7.7.

</details>

### Приклад 7.6. Те саме з `aiohttp`

Інший інтерфейс: відповідь — асинхронний контекстний менеджер, тіло читаємо через `await`.

In [ ]:
async def check_all_aiohttp(ids, delay=0.5):
    async with aiohttp.ClientSession(timeout=aiohttp.ClientTimeout(total=5)) as session:
        async def fetch(restaurant_id):
            url = f"{BASE}/restaurants/{restaurant_id}/status"
            async with session.get(url, params={"delay": delay}) as response:
                response.raise_for_status()
                return await response.json()

        return await asyncio.gather(*(fetch(restaurant_id) for restaurant_id in ids))

In [ ]:
start = time.perf_counter()
statuses = await check_all_aiohttp(range(1, 6))
print(f"aiohttp: {time.perf_counter() - start:.1f} с", [status["name"] for status in statuses])

### Приклад 7.7. Обмежуємо одночасність: `asyncio.Semaphore`

10 запитів (кожен ресторан двічі) по 0,3 с. Без обмеження — усі разом. З `Semaphore(3)` — не більше трьох одночасно: 4 «хвилі» по 0,3 с.

In [ ]:
async def check_limited(ids, limit, delay=0.3):
    semaphore = asyncio.Semaphore(limit)
    async with httpx.AsyncClient(base_url=BASE, timeout=5) as client:
        async def fetch(restaurant_id):
            async with semaphore:                    # чекаємо, поки звільниться «місце»
                response = await client.get(f"/restaurants/{restaurant_id}/status", params={"delay": delay})
                return response.json()

        return await asyncio.gather(*(fetch(restaurant_id) for restaurant_id in ids))

In [ ]:
ids = list(range(1, 6)) * 2
for limit in [10, 3, 1]:
    start = time.perf_counter()
    results = await check_limited(ids, limit)
    print(f"limit={limit:2}: {len(results)} відповідей за {time.perf_counter() - start:.1f} с")

`limit=1` — це знову «по черзі» (3 с), `limit=10` — усі разом (0,3 с). На справжньому API ліміт беруть з документації. Інший спосіб — обмежити з'єднання самого клієнта: `httpx.AsyncClient(limits=httpx.Limits(max_connections=3))`.

### Приклад 7.8. Одна помилка серед багатьох: `return_exceptions=True`

Без нього `gather` кидає перший виняток і втрачає решту результатів. З ним — повертає винятки **на місці результатів**.

In [ ]:
async def check_all_safe(ids):
    async with httpx.AsyncClient(base_url=BASE, timeout=5) as client:
        async def fetch(restaurant_id):
            response = await client.get(f"/restaurants/{restaurant_id}/status")
            response.raise_for_status()
            return response.json()["name"]

        return await asyncio.gather(*(fetch(restaurant_id) for restaurant_id in ids), return_exceptions=True)

In [ ]:
for restaurant_id, result in zip([1, 42, 3], await check_all_safe([1, 42, 3])):
    if isinstance(result, Exception):
        print(restaurant_id, "→ помилка:", type(result).__name__, result.response.status_code)
    else:
        print(restaurant_id, "→", result)

| | `requests` | `httpx` | `aiohttp` |
|---|---|---|---|
| Синхронний клієнт | так | так (`Client`) | ні |
| Асинхронний клієнт | ні | так (`AsyncClient`) | так (`ClientSession`) |
| Тайм-аут за замовчуванням | **немає** | 5 с | 5 хвилин |
| Де зустрінеш | скрипти, більшість прикладів | FastAPI-проєкти, тести, SDK | старіші async-проєкти, боти |

---
## 8. Свій клієнт до API

Коли запити розкидані по всій програмі, кожного разу повторюється те саме: адреса, токен, тайм-аут, перевірка статусу, повтори. Тому роботу з API збирають в **один клас-клієнт**, а решта програми отримує звичайні словники і **свої** винятки.

```text
логіка диспетчерської ── api.create_order(...) ──► SmachnoClient
                                                     │ адреса, токен, тайм-аут
                                                     │ повтори лише для GET
                                                     ▼
                                                 httpx.Client ── HTTP ──► API ресторанів
                                                     │
            NotFound / ValidationFailed / dict  ◄────┘ коди → винятки
```

### Приклад 8.1. Свої винятки

In [ ]:
class SmachnoError(Exception):
    """Будь-яка помилка API ресторанів."""


class NotFound(SmachnoError):
    pass


class ValidationFailed(SmachnoError):
    def __init__(self, errors):
        super().__init__("; ".join(errors))
        self.errors = errors

In [ ]:
error = ValidationFailed(["customer: обов'язкове поле", "total: число більше за 0"])
print(str(error))
print(error.errors)
print(isinstance(error, SmachnoError), isinstance(NotFound("x"), SmachnoError))

Ієрархія (урок 14) дає вибір: `except NotFound` — лише «немає», `except SmachnoError` — будь-яка помилка API.

### Приклад 8.2. Клієнт: адреса, токен, тайм-аут, повтори, коди → винятки

In [ ]:
class SmachnoClient:
    def __init__(self, base_url, token, timeout=5, retries=3, pause=0.2):
        self._http = httpx.Client(base_url=base_url, timeout=timeout,
                                  headers={"Authorization": f"Bearer {token}"})
        self._retries = retries
        self._pause = pause

    def __enter__(self):
        return self

    def __exit__(self, *exc):
        self._http.close()

    def _request(self, method, path, **kwargs):
        can_retry = method == "GET"                  # POST не повторюємо: можна створити дубль
        attempts = self._retries if can_retry else 1
        for attempt in range(1, attempts + 1):
            try:
                response = self._http.request(method, path, **kwargs)
            except httpx.TransportError as error:
                if attempt == attempts:
                    raise SmachnoError(f"{method} {path}: {type(error).__name__}") from error
            else:
                if response.status_code < 500 or attempt == attempts:
                    break
            time.sleep(self._pause * 2 ** (attempt - 1))

        if response.status_code == 404:
            raise NotFound(response.json()["error"])
        if response.status_code == 422:
            raise ValidationFailed(response.json()["errors"])
        if response.is_error:
            raise SmachnoError(f"{method} {path}: {response.status_code}")
        return response.json()

    def restaurants(self, district=None):
        params = {"district": district} if district else {}
        return self._request("GET", "/restaurants", params=params)

    def restaurant(self, restaurant_id):
        return self._request("GET", f"/restaurants/{restaurant_id}")

    def create_order(self, restaurant_id, customer, total):
        order = {"restaurant_id": restaurant_id, "customer": customer, "total": total}
        return self._request("POST", "/orders", json=order)

### Приклад 8.3. Читання і `NotFound`

In [ ]:
with SmachnoClient(BASE, token) as api:
    print([restaurant["name"] for restaurant in api.restaurants("Оболонь")])
    print(len(api.restaurants()), "ресторанів усього")
    try:
        api.restaurant(42)
    except NotFound as error:
        print("NotFound:", error)

### Приклад 8.4. Погані дані → `ValidationFailed` зі списком помилок

In [ ]:
requests.post(f"{BASE}/reset", timeout=5)
with SmachnoClient(BASE, token) as api:
    try:
        api.create_order(1, "", 100)
    except ValidationFailed as error:
        print("ValidationFailed:", error.errors)
    print(api.create_order(2, "Марко", 300))

### Приклад 8.5. Неправильний токен → `SmachnoError`, але не `NotFound`

In [ ]:
with SmachnoClient(BASE, "wrong-token") as api:
    try:
        api.create_order(2, "Марко", 300)
    except NotFound:
        print("сюди не потрапимо: 401 — це не NotFound")
    except SmachnoError as error:
        print(type(error).__name__, "—", error)

Порядок `except` важливий: спершу вужчі класи (`NotFound`), потім ширший (`SmachnoError`). 401 не має свого класу, тому потрапив у загальний.

### Приклад 8.6. Повтори всередині клієнта

`/flaky` двічі відповідає 503. Клієнт сам повторить `GET` — код, що його викликає, побачить лише успіх. А `POST` клієнт не повторює ніколи.

In [ ]:
requests.post(f"{BASE}/reset", timeout=5)
with SmachnoClient(BASE, token, retries=3, pause=0.05) as api:
    print(api._request("GET", "/flaky"))

requests.post(f"{BASE}/reset", timeout=5)
with SmachnoClient(BASE, token, retries=2, pause=0.05) as api:
    try:
        api._request("GET", "/flaky")
    except SmachnoError as error:
        print("здались після 2 спроб:", error)

**🔮 Прогноз:** Чому в клієнті повтори лише для `GET`, а не для `create_order`?

<details>
<summary>Відповідь</summary>

Якщо `POST /orders` не дочекався відповіді, замовлення могло вже створитися. Автоматичний повтор створить дубль (приклад 4.5). Для таких випадків API дають **ключ ідемпотентності** (урок 33).

</details>

Так влаштовані офіційні клієнти великих сервісів (Stripe, Telegram, OpenAI): усередині — HTTP, назовні — методи й винятки. Замінити `httpx` на `requests` — правка в одному класі.

---
## 9. Практика: замовлення з черги

### Приклад 9.1. Відправити нічні замовлення і скласти звіт

За ніч надійшли замовлення з сайту; частина — з помилками. Треба відправити всі і скласти звіт: що створено, що відхилено і чому. Помилка в одному замовленні не повинна зупиняти решту.

In [ ]:
night_orders = [
    {"restaurant_id": 1, "customer": "Ірина", "total": 380},
    {"restaurant_id": 7, "customer": "Богдан", "total": 250},
    {"restaurant_id": 3, "customer": "", "total": 0},
    {"restaurant_id": 5, "customer": "Софія", "total": 190},
]


def send_orders(api, orders):
    created, rejected = [], []
    for order in orders:
        try:
            result = api.create_order(**order)
        except ValidationFailed as error:
            rejected.append((order["customer"] or "?", error.errors))
        else:
            created.append(result["id"])
    return created, rejected

In [ ]:
requests.post(f"{BASE}/reset", timeout=5)
with SmachnoClient(BASE, token) as api:
    created, rejected = send_orders(api, night_orders)
print("створено:", created)
for customer, errors in rejected:
    print("відхилено:", customer, errors)

- `create_order(**order)` — ключі словника стають іменованими аргументами (урок 8).
- `except ValidationFailed` ловить лише «погані дані». Якщо API впаде (`SmachnoError`), виняток полетить далі: звіт на половині даних гірший за чесну помилку.
- `try / except / else`: `else` виконується, лише якщо винятку не було.

### Задача 3. Лише свій район

Напиши `send_orders_in(api, orders, district=None)` — як `send_orders`, але повертає **три** списки: `created`, `rejected`, `skipped`.

- якщо `district` задано — отримай ресторани району **одним** викликом `api.restaurants(district)` (до циклу) і відправляй лише замовлення до них; решту замовлень (словники як є) поклади в `skipped`;
- без `district` — поведінка як у `send_orders`, `skipped` порожній, `api.restaurants` не викликається.

Перевірка — у наступній клітинці.

In [ ]:
def send_orders_in(api, orders, district=None):
    # YOUR CODE HERE
    pass


In [ ]:
class CountingClient(SmachnoClient):
    """Клієнт, що рахує виклики restaurants() — щоб перевірити «один запит на весь виклик»."""
    calls = 0

    def restaurants(self, district=None):
        self.calls += 1
        return super().restaurants(district)


requests.post(f"{BASE}/reset", timeout=5)
with CountingClient(BASE, token) as api:
    result = send_orders_in(api, night_orders, district="Поділ")
    assert result is not None, "функція має повертати (created, rejected, skipped)"
    created, rejected, skipped = result
    assert created == [101], created
    assert rejected == [], rejected
    assert [order["restaurant_id"] for order in skipped] == [7, 3, 5], skipped
    assert api.calls == 1, f"api.restaurants викликано {api.calls} разів — потрібен один запит до циклу"

requests.post(f"{BASE}/reset", timeout=5)
with CountingClient(BASE, token) as api:
    created, rejected, skipped = send_orders_in(api, night_orders)
    assert created == [101, 102] and len(rejected) == 2 and skipped == [], (created, rejected, skipped)
    assert api.calls == 0, "без district список ресторанів не потрібен"
print("✅ Задача 3 розв'язана")

---
## 10. Справжній API: PyPI

[PyPI](https://pypi.org/) — каталог пакетів, звідки `pip` їх встановлює, — має відкритий [JSON API](https://docs.pypi.org/api/json/): `https://pypi.org/pypi/<пакет>/json` → поле `info.version`; неіснуючий пакет — 404.

Щоб приклади працювали й без інтернету, спершу зробимо **офлайн-двійника**: `httpx.MockTransport` підставляє замість мережі звичайну функцію, яка отримує запит і повертає відповідь. Запит не виходить за межі Python — так роблять у тестах (урок 42).

### Приклад 10.1. Підробний PyPI через `httpx.MockTransport`

In [ ]:
FAKE_VERSIONS = {"requests": "2.32.3", "httpx": "0.28.1"}      # умовні версії для офлайн-прикладу


def fake_pypi(request):
    """Обробник замість мережі: отримує httpx.Request, повертає httpx.Response."""
    parts = request.url.path.strip("/").split("/")             # ['pypi', 'requests', 'json']
    name = parts[1] if len(parts) == 3 else ""
    if name not in FAKE_VERSIONS:
        return httpx.Response(404, json={"message": "Not Found"})
    return httpx.Response(200, json={"info": {"name": name, "version": FAKE_VERSIONS[name]}})


FAKE_PYPI = httpx.MockTransport(fake_pypi)

In [ ]:
with httpx.Client(base_url="https://pypi.org", transport=FAKE_PYPI) as client:
    for name in ["requests", "no-such-package-xyz-31"]:
        response = client.get(f"/pypi/{name}/json")
        print(response.status_code, response.json())

Код клієнта той самий, що для справжнього `pypi.org`, змінився лише `transport`.

### Приклад 10.2. Останні версії кількох пакетів — одночасно

Корутина для одного пакета повертає пару `(назва, версія)`, `dict(await asyncio.gather(...))` складає з пар словник. 404 перевіряємо **до** `raise_for_status()`: «немає пакета» — це `None`, а інші помилки мають вилетіти винятком. Параметр `transport` дозволяє підставити двійника.

In [ ]:
async def latest_versions(packages, transport=None):
    async with httpx.AsyncClient(base_url="https://pypi.org", timeout=10, transport=transport) as client:
        async def fetch(name):
            response = await client.get(f"/pypi/{name}/json")
            if response.status_code == 404:
                return name, None
            response.raise_for_status()
            return name, response.json()["info"]["version"]

        return dict(await asyncio.gather(*(fetch(name) for name in packages)))

In [ ]:
offline = await latest_versions(["requests", "httpx", "no-such-package-xyz-31"], transport=FAKE_PYPI)
print("офлайн:", offline)
assert offline == {"requests": "2.32.3", "httpx": "0.28.1", "no-such-package-xyz-31": None}

### Приклад 10.3. Живий запит до pypi.org

Та сама функція без `transport` — справжня мережа і HTTPS. Без інтернету `httpx` кине `TransportError` (`ConnectError` чи `ConnectTimeout`), і клітинка надрукує пояснення.

In [ ]:
try:
    live = await latest_versions(["requests", "httpx", "no-such-package-xyz-31"])
except httpx.TransportError as error:
    print("⚠️ Немає доступу до pypi.org — приклад пропущено:", type(error).__name__)
else:
    print("pypi.org:", live)
    assert live["no-such-package-xyz-31"] is None

Адреса інша — `https://`, з'єднання шифроване (TLS), — а код той самий. Сертифікат сервера бібліотека перевіряє сама; `verify=False` «лікує» помилку сертифіката ціною захисту — так не роблять.

---
## 11. Знайди помилку

### Приклад 11.1. Функція з двома проблемами

In [ ]:
def restaurant_name_buggy(restaurant_id):
    response = requests.get(f"{BASE}/restaurants/{restaurant_id}")
    return response.json()["name"]

In [ ]:
print(restaurant_name_buggy(2))
try:
    restaurant_name_buggy(42)
except KeyError as error:
    print("KeyError:", error)

Для 42 функція падає з `KeyError: 'name'` — повідомлення нічого не каже про справжню причину. Проблем **дві**:

1. **статус не перевірено**: на 404 сервер повернув `{"error": …}`, а ми шукаємо `"name"`;
2. **немає `timeout`**: у день, коли сервер зависне, функція чекатиме вічно — і разом з нею вся диспетчерська.

### Приклад 11.2. Виправлено

In [ ]:
def restaurant_name(restaurant_id):
    response = requests.get(f"{BASE}/restaurants/{restaurant_id}", timeout=5)
    response.raise_for_status()
    return response.json()["name"]

In [ ]:
print(restaurant_name(2))
try:
    restaurant_name(42)
except requests.HTTPError as error:
    print("HTTPError:", error)

Тепер для 42 одразу видно, що сталося: `404 Client Error: Not Found for url: …`.

---
## 12. Зупиняємо сервер

### Приклад 12.1. Сервер зупинено — клієнт бачить `ConnectionError`

Сервер працював у фоновому потоці ядра. Зупиняємо його й звільняємо порт; після цього той самий запит отримує відмову в з'єднанні. Щоб знову запустити — перезапусти клітинку з прикладу 0.1.

In [ ]:
if smachno_api._server is not None:
    smachno_api._server.shutdown()
    smachno_api._server.server_close()
    smachno_api._server = None
try:
    requests.get(f"{BASE}/restaurants", timeout=1)
except requests.ConnectionError:
    print("сервер зупинено, порт", PORT, "вільний")

---
## Самоперевірка

1. Які кроки між `requests.get("https://pypi.org/...")` і словником у Python?
2. Що поверне `requests` на 404 — виняток чи відповідь?
3. Чому `timeout` обов'язковий у `requests`? Який він у `httpx`?
4. Що можна повторювати, а що ні?
5. Чим `json=` відрізняється від `data=`?
6. Чому 5 запитів з `gather` займають стільки ж, скільки один? Як обмежити одночасність?
7. Навіщо писати свій клієнт-клас до API?

<details>
<summary>Відповіді</summary>

1. URL → DNS (ім'я → IP) → TCP-з'єднання на порт 443 → TLS → HTTP-запит і відповідь → `json()`.
2. Відповідь зі `status_code == 404`; виняток — лише після `raise_for_status()`.
3. Без нього `requests` чекатиме завислий сервер вічно. У `httpx` за замовчуванням 5 секунд.
4. 5xx, тайм-аути, збої мережі — для ідемпотентних запитів (GET), обмежену кількість разів з паузою, що росте. Не повторюємо 4xx і автоматично — `POST`.
5. `json=` — тіло JSON і `Content-Type: application/json`; `data=словник` — форма `application/x-www-form-urlencoded`.
6. Запити не рахують, а чекають мережу; поки чекає один, цикл подій відправляє інші. Обмежують `asyncio.Semaphore` або `httpx.Limits`.
7. Адреса, токен, тайм-аут, повтори й перетворення кодів на винятки — в одному місці; решта програми не знає про HTTP, а в тестах клієнт легко підмінити.

</details>

## Далі

- Книга: [Урок 32](https://nikoriakviktor.github.io/PY-Course-Victor-Nikoriak-22-09-2026/modules/m4/lesson_32/) — DNS, порти, TCP і TLS, `Session`, ієрархія винятків, архітектура клієнта, `requests` vs `httpx` vs `aiohttp`.
- Документація: [requests](https://requests.readthedocs.io/en/latest/user/quickstart/), [httpx](https://www.python-httpx.org/quickstart/), [aiohttp client](https://docs.aiohttp.org/en/stable/client_quickstart.html).
- **Урок 33** — REST: принципи дизайну API — той самий обмін, але з боку того, хто проєктує сервер.